# Security Alert Classifier — one alert, two cloud LLMs
### Introduction to AI · IT7075: Applied AI for Cybersecurity

Your first hands-on example. You send the **same** security alert to two hosted language models — **Claude** (Anthropic) and **OpenAI** — and compare how they classify it. The point is not which model is "right," but to *feel* what an LLM does with a security question and to see that two models can differ.

**Runs anywhere** — locally, or in Google Colab. Every model call is **guarded**: if a key is missing, that step prints a notice and is skipped, so the notebook runs with one key, both, or neither.

> **Keys are secrets.** Never paste an API key into a cell. Use Colab **Secrets** (the key icon) or a hidden **.env** file, as shown below. A leaked key spends real money — rotate it immediately if exposed.

## 0 — Install the libraries (Colab)
Skip this if they are already installed locally.

In [1]:
# In Colab, uncomment and run once:
# %pip install anthropic openai python-dotenv

## 1 — Provide your keys (safely)
This cell loads keys **without hard-coding them**. It tries, in order: Colab Secrets, then a hidden `.env` file, then existing environment variables. It prints only whether a key was *found* — never the key itself.

In [2]:
import os

# (a) Colab Secrets: add OPENAI_API_KEY / ANTHROPIC_API_KEY via the key icon, then:
try:
    from google.colab import userdata
    for k in ('ANTHROPIC_API_KEY', 'OPENAI_API_KEY'):
        v = userdata.get(k)
        if v:
            os.environ[k] = v
except Exception:
    pass

# (b) Local .env (searched upward from here); no-op in Colab / if none:
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
except Exception:
    pass

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
HAS_ANTHROPIC = bool(ANTHROPIC_API_KEY)
HAS_OPENAI = bool(OPENAI_API_KEY)

# Model IDs in one place — swap if a model is unavailable to you:
CLAUDE_MODEL = 'claude-haiku-4-5-20251001'
OPENAI_MODEL = 'gpt-5-mini'

print('Anthropic key:', 'found' if HAS_ANTHROPIC else 'MISSING (Claude step will skip)')
print('OpenAI key:   ', 'found' if HAS_OPENAI else 'MISSING (OpenAI step will skip)')

Anthropic key: MISSING (Claude step will skip)
OpenAI key:    found


## 2 — The alert and the prompt
One realistic alert, and one instruction we will give both models. Read the alert: five failed admin logins from one IP in 20 seconds, then a success. What does your gut say?

In [3]:
alert = "5 failed logins for 'admin' from 203.0.113.7 in 20s, then 1 success"

PROMPT = f"Classify this security alert as benign or suspicious and explain in one sentence: {alert}"
print(PROMPT)

Classify this security alert as benign or suspicious and explain in one sentence: 5 failed logins for 'admin' from 203.0.113.7 in 20s, then 1 success


## 3 — Ask Claude
Anthropic's Messages API. Guarded: skips if no Anthropic key.

In [4]:
import textwrap

def classify_with_claude(alert):
    if not HAS_ANTHROPIC:
        return '[skipped — no ANTHROPIC_API_KEY set]'
    from anthropic import Anthropic
    client = Anthropic(api_key=ANTHROPIC_API_KEY)
    msg = client.messages.create(
        model=CLAUDE_MODEL, max_tokens=150,
        messages=[{'role': 'user', 'content': PROMPT}])
    return msg.content[0].text.strip()

claude_result = classify_with_claude(alert)
print(textwrap.fill(claude_result, width=80))

[skipped — no ANTHROPIC_API_KEY set]


## 4 — Ask OpenAI
OpenAI's Chat Completions API. Guarded: skips if no OpenAI key.

In [5]:
def classify_with_openai(alert):
    if not HAS_OPENAI:
        return '[skipped — no OPENAI_API_KEY set]'
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[{'role': 'user', 'content': PROMPT}])
    return response.choices[0].message.content.strip()

openai_result = classify_with_openai(alert)
print(textwrap.fill(openai_result, width=80))

Suspicious — five rapid failed attempts for user "admin" from 203.0.113.7
followed immediately by a success strongly indicates brute-force or credential-
stuffing activity and a likely account compromise.


## 5 — Compare the two verdicts
Side by side. Do they agree? Is one more specific, or better justified? Both should flag this as **suspicious** (a likely brute-force that finally succeeded) — but notice the differences in wording and reasoning.

In [6]:
def panel(title, body):
    print('-' * 70); print(title); print('-' * 70)
    print(textwrap.fill(body, width=70)); print()

print('ALERT:', alert, '\n')
panel(f'Claude  ({CLAUDE_MODEL})', claude_result)
panel(f'OpenAI  ({OPENAI_MODEL})', openai_result)
print('Reminder: treat each verdict as a DRAFT to verify, not a fact to trust.')

ALERT: 5 failed logins for 'admin' from 203.0.113.7 in 20s, then 1 success 

----------------------------------------------------------------------
Claude  (claude-haiku-4-5-20251001)
----------------------------------------------------------------------
[skipped — no ANTHROPIC_API_KEY set]

----------------------------------------------------------------------
OpenAI  (gpt-5-mini)
----------------------------------------------------------------------
Suspicious — five rapid failed attempts for user "admin" from
203.0.113.7 followed immediately by a success strongly indicates
brute-force or credential-stuffing activity and a likely account
compromise.

Reminder: treat each verdict as a DRAFT to verify, not a fact to trust.


## 6 — Your turn
Change `my_alert` to something else and re-run. Try a clearly **benign** event and an **ambiguous** one, and see whether the models agree and how confident they sound.

Ideas: `"User connected to the VPN from a new country at 3am"` · `"Daily backup job completed successfully at 02:00"` · `"PowerShell downloaded and ran a script from a pastebin URL"`

In [7]:
my_alert = 'User connected to the VPN from a new country at 3am'
my_prompt = f'Classify this security alert as benign or suspicious and explain in one sentence: {my_alert}'

# reuse the helpers with the new prompt
PROMPT = my_prompt
print('ALERT:', my_alert, '\n')
panel('Claude', classify_with_claude(my_alert))
panel('OpenAI', classify_with_openai(my_alert))

ALERT: User connected to the VPN from a new country at 3am 

----------------------------------------------------------------------
Claude
----------------------------------------------------------------------
[skipped — no ANTHROPIC_API_KEY set]

----------------------------------------------------------------------
OpenAI
----------------------------------------------------------------------
Suspicious — a VPN connection from a new country at 3am is an
anomalous behavior that could indicate compromised credentials or
unauthorized access and should be investigated (verify the user’s
travel, recent MFA/ password events, and session activity).

